[![Open HW01 in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skgallagher/stat-methods-ai-public/blob/main/homeworks/hw01.ipynb)

# Homework 01 — Images as Data and Baseline Evaluation

**Submit:** export this completed notebook as a PDF and submit to Gradescope (`HW01 (PDF)`).

**Optional:** also submit the completed `.ipynb` to `HW01 (Notebook - optional)`.

This homework extends the Week 1 MNIST lab. Your goal is not to build the best digit classifier. Your goal is to practice treating a dataset, a model, and a benchmark result as statistical evidence.

## Readings

- Leo Breiman (2001), "Statistical Modeling: The Two Cultures"
- Robert E. Kass (2021), ["The Two Cultures: Statistics and Machine Learning in Science"](https://www.stat.cmu.edu/~kass/papers/KassOnBreiman.pdf)

As you read, do not frame the papers as two teams to choose between. Focus on what each paper says statistical reasoning is for.

## Setup

Run the setup cell. If Colab reports that a package is missing, uncomment the install line and run the cell again.

In [ ]:
# !pip -q install numpy scikit-learn torch torchvision matplotlib plotly seaborn

import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

import torch
import torchvision
import torchvision.transforms as T

np.random.seed(0)
torch.manual_seed(0)

def show_confusion(y_true, y_pred, labels=None, title="Confusion matrix"):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig = px.imshow(cm, text_auto=True, aspect="auto",
                    labels=dict(x="Predicted", y="True", color="Count"),
                    title=title)
    fig.show()
    return cm

def show_examples(X_values, y_true, y_pred, indices, title, max_images=12):
    if len(indices) == 0:
        print("No examples to show.")
        return
    indices = indices[:max_images]
    cols = min(6, len(indices))
    rows = int(np.ceil(len(indices) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, idx in zip(axes, indices):
        ax.imshow(X_values[idx].reshape(28, 28), cmap="gray")
        ax.set_title(f"true={y_true[idx]}, pred={y_pred[idx]}")
        ax.axis("off")
    for ax in axes[len(indices):]:
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


## Load MNIST

Use the same subset size as lab unless you have a reason to change it. The fixed random seed makes your results reproducible.

In [ ]:
transform = T.Compose([T.ToTensor()])
train_ds = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)

n = 12000
xs = torch.stack([train_ds[i][0] for i in range(n)])
X = xs.view(n, -1).numpy()
y = np.array([train_ds[i][1] for i in range(n)])

print("X shape:", X.shape)
print("y shape:", y.shape)


## Problem 1 — EDA: What Data Do We Have?

Create:

- a class-balance plot
- mean and variance images
- one PCA plot

Then interpret what these summaries reveal about the dataset. What aspects of the data seem natural, and what aspects seem shaped by preprocessing or curation?

In [ ]:
# Class balance
counts = np.bincount(y, minlength=10)
fig = px.bar(x=list(range(10)), y=counts,
             labels={"x": "Digit", "y": "Count"},
             title="MNIST class balance (subset)")
fig.show()

# Mean and variance images
mean_img = xs.mean(dim=0).squeeze(0).numpy()
var_img = xs.var(dim=0).squeeze(0).numpy()

fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].imshow(mean_img, cmap="gray")
ax[0].set_title("Mean image")
ax[0].axis("off")
ax[1].imshow(var_img, cmap="magma")
ax[1].set_title("Pixel variance")
ax[1].axis("off")
plt.tight_layout()
plt.show()

# PCA
pca = PCA(n_components=20, random_state=0)
Z = pca.fit_transform(X)
fig = px.scatter(x=Z[:, 0], y=Z[:, 1], color=y.astype(str),
                 labels={"x": "PC1", "y": "PC2", "color": "Digit"},
                 title="MNIST PCA (PC1 vs PC2)")
fig.show()


**Answer 1.** Write 1-2 paragraphs interpreting your EDA.

> Replace this text with your answer.

## Problem 2 — Baselines: What Evidence Does Each Model Give?

Fit multinomial logistic regression and a random forest on the same training/validation split.

Report validation accuracy for each model, the difference in accuracy, and one confusion matrix for each model. Then compare what each model helps you learn about the task.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

logit = LogisticRegression(max_iter=200, multi_class="multinomial", solver="lbfgs")
logit.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=100, max_depth=16, random_state=0, n_jobs=-1)
rf.fit(X_train, y_train)

pred_logit = logit.predict(X_val)
pred_rf = rf.predict(X_val)

acc_logit = accuracy_score(y_val, pred_logit)
acc_rf = accuracy_score(y_val, pred_rf)

print(f"Logistic regression validation accuracy: {acc_logit:.3f}")
print(f"Random forest validation accuracy:       {acc_rf:.3f}")
print(f"Difference (RF - logit):                 {acc_rf - acc_logit:.3f}")

show_confusion(y_val, pred_logit, labels=list(range(10)), title="Logistic regression confusion matrix")
show_confusion(y_val, pred_rf, labels=list(range(10)), title="Random forest confusion matrix")


**Answer 2.** Discuss the two baselines. Do not simply say which model is "better." Explain what each model helps you see, and name 1-2 specific digit confusions that are common or surprising.

> Replace this text with your answer.

## Problem 3 — Math/Theory: Accuracy as an Estimate

For a fixed fitted classifier $f$, define the target accuracy on a population of future examples as

$$
p = P(f(X) = Y).
$$

On a validation set of size $m$, the validation accuracy is

$$
\hat p = \frac{1}{m}\sum_{i=1}^m I(f(x_i)=y_i).
$$

Answer the following written questions.

1. Under what assumptions is $\hat p$ an unbiased or approximately unbiased estimate of $p$?
2. Why is $\hat p$ not the same thing as the model's true deployed accuracy?
3. Treating the validation outcomes as independent Bernoulli trials, derive or state the approximate standard error of $\hat p$.
4. Compute an approximate 95% confidence interval for each model's validation accuracy.
5. Based on these intervals, how cautious should we be about saying one model is truly better than the other?

This is a deliberately simple uncertainty calculation. Later in the course we will refine it, especially because the two models are evaluated on the same validation examples.

In [ ]:
m = len(y_val)

def wald_ci(acc, m, z=1.96):
    se = np.sqrt(acc * (1 - acc) / m)
    return acc - z * se, acc + z * se, se

for name, acc in [("Logistic regression", acc_logit), ("Random forest", acc_rf)]:
    lo, hi, se = wald_ci(acc, m)
    print(f"{name}: accuracy={acc:.3f}, SE={se:.4f}, 95% CI=({lo:.3f}, {hi:.3f})")


**Answer 3.** Show your reasoning, not just the computed interval. Be explicit about the assumptions behind the calculation and at least one reason those assumptions might be imperfect here.

> Replace this text with your answer.

## Problem 4 — Error Analysis

Inspect individual validation examples. Choose at least three images from the displays below and discuss whether the mistakes look like data issues, model issues, or ambiguous labels.

In [ ]:
both_wrong = np.where((pred_logit != y_val) & (pred_rf != y_val))[0]
logit_wrong_rf_right = np.where((pred_logit != y_val) & (pred_rf == y_val))[0]
rf_wrong_logit_right = np.where((pred_rf != y_val) & (pred_logit == y_val))[0]

print("Both wrong:", len(both_wrong))
print("Logit wrong, RF right:", len(logit_wrong_rf_right))
print("RF wrong, logit right:", len(rf_wrong_logit_right))

show_examples(X_val, y_val, pred_rf, both_wrong, "Examples both models missed; title uses RF prediction")
show_examples(X_val, y_val, pred_logit, logit_wrong_rf_right, "Logistic regression wrong, RF right")
show_examples(X_val, y_val, pred_rf, rf_wrong_logit_right, "RF wrong, logistic regression right")


**Answer 4.** Discuss at least three individual errors. What did looking at actual images teach you that accuracy and confusion matrices did not?

> Replace this text with your answer.

## Problem 5 — Benchmark Critique

Write 2-3 paragraphs critiquing MNIST as evidence about digit-recognition systems.

Address:

- What target population might MNIST reasonably represent?
- What target population does it probably not represent?
- What are two sources of bias, curation, or mismatch?
- What additional dataset, perturbation, or evaluation would make the evidence stronger?

**Answer 5.**

> Replace this text with your answer.

## Problem 6 — Reading Reflection: Breiman and Kass

In 1-2 paragraphs, explain how Kass's critique changes or complicates the simple "two cultures" framing.

A strong answer will avoid treating logistic regression as "statistics" and random forest as "not statistics." Instead, discuss what kinds of questions, assumptions, and evidence each modeling approach makes easier or harder to see.

**Answer 6.**

> Replace this text with your answer.

## Problem 7 — AI Checkpoint: Ask, Then Critique

Ask an AI tool the following prompt:

> "MNIST is a good benchmark for evaluating whether an AI system can recognize handwritten digits. Explain why, using statistical reasoning."

Paste the AI's answer below or summarize it briefly. Then critique it by identifying:

- one claim the AI made that is reasonable
- one claim that is incomplete, overstated, or missing important context
- one follow-up question you would ask before trusting MNIST as evaluation evidence

**AI output or summary.**

> Replace this text with the AI output or a brief summary.

**Critique.**

> Replace this text with your critique.

## Submission Checklist

Before exporting to PDF, make sure your notebook includes:

- all requested plots and accuracy results
- written answers for Problems 1-7
- enough code and explanation that your work is reproducible

In Colab: **File -> Print -> Save as PDF**.